In [ ]:
# Purpose - compare spectra from multiple files

findices = [9, 100]

In [ ]:
# Imports and Folder Selection

%load_ext autoreload
%autoreload 2

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun
from FieldLineHelper import FieldLineHelper 

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

runs = [TristanRun(findex) for findex in findices]
boilerplate = "TODO" #run.boilerplate + f"tstep = {run.steps[0]}"

datas = [run.config_data(run.steps, mode = "spectra") for run in runs]

In [ ]:
d = datas[0]
print(d.spectra.isel(t = 60))

In [ ]:
def plot_spectrum(subplot, gammas, run):
    log_bins = 10 ** np.linspace(-4, 4, 100) #300
    hist, bins = np.histogram((gammas - 1) / run.sigma, bins = log_bins, density = True)

    subplot.set_xlabel("$(\gamma - 1) / \sigma$")
    subplot.set_ylabel("$dN / d\gamma$")
    subplot.set_title(f"Electron Spectra")
    subplot.grid(visible = True)
    subplot.axvline(1, color = 'black', linestyle = '--', label = '$\gamma - 1 = \sigma$', alpha = 0.5)
    subplot.axvline(1 / run.sigma, color = 'orange', linestyle = '--', label = '$\gamma - 1 = 1$', alpha = 0.5)

    outflow_gamma = (1 - run.beta_A ** 2) ** -0.5
    subplot.axvline((outflow_gamma - 1) / run.sigma, color = 'purple', linestyle = '--', label = '$\gamma = \gamma_A$', alpha = 0.5)

    subplot.legend()
    
    subplot.loglog(bins[1:], hist)

In [ ]:
def save_plot(name, dpi = 200):
    plt.savefig(f"/scratch/gpfs/ar3676/analysis/plots/{name}.png", dpi = dpi)

In [ ]:
# Plot Domain and Spectrum

species = 1

plt.close()
fig, ax1 = plt.subplots(nrows = 1, ncols = 1, figsize = (6, 4))

colors = plt.cm.viridis(np.linspace(0, 1, len(runs)))
colors = ['red', 'blue']

tot_normalize = True

for i, run in enumerate(runs):
    label = True
    xcoeff = 1 / run.sigma

    d = run.config_data(run.steps, mode = "spectra")

    bs = d.spectra.isel(t = run.steps[-1])
    print(bs)

    ebins, spectrum = TristanRun.get_spectrum(d, run.steps[-1], species, normalized = tot_normalize)

    # integrate spectrum over ebins using sensible edge handling
    e = ebins
    edges = np.empty(e.size + 1)
    edges[1:-1] = 0.5 * (e[1:] + e[:-1])
    edges[0] = e[0] - (edges[1] - e[0])
    edges[-1] = e[-1] + (e[-1] - edges[-2])
    widths = np.diff(edges)

    integral = np.sum(spectrum * widths)
    print("Integrated spectrum over ebins:", integral)

    if (label):
        ax1.loglog(ebins * xcoeff, spectrum * run.sigma, linewidth = 1,
                markersize = 1.5, label = f'$\sigma = $ {run.sigma}')

    else:
        ax1.loglog(bs.ebins * xcoeff, bs.calc_hist() * bs.total, linewidth = 1,
                markersize = 1.5, color=colors[i])

plt.suptitle("")

run = runs[0]

indicator_line = run.beta_A ** 2 / 0.3

ax1.vlines(indicator_line, color="#1f77b4", linestyle='--', alpha=0.7, ymin=0, ymax=0.339)

title = "Electron Spectra"

#for i in range(len(runs)):
    #title += f"\n {runs[i].code}, $\gamma_{{mean}} - 1 = {means[i]:.2f}, p = {probs[i]:.2e}$"

ax1.set_title(title)
ax1.grid(visible = True)
ax1.set_xlim([1e-4, 1e2])

#ax1.axvspan(interval[0], interval[1], color='yellow', alpha=0.3, label='Interval')

#ax2.set_ylim([1, 1e12])
ax1.legend()
#ax2.loglog(spec_bins, spec_counts, linewidth = 1, markersize = 1.5)

plt.xlabel("$(\gamma - 1) / \sigma$")  

if (tot_normalize):
    plt.ylabel("$f(\gamma)$")
else:
    plt.ylabel("$dN / d\gamma$")

plt.xlim([1e-3, 1e2])

plt.tight_layout()
#save_polished_fig("spectra", dpi = 400)

plt.show()

In [ ]:
print(bs.n1)


In [ ]:
print(run.spec_nx)
print(run.spec_ny)

In [ ]:
print(0 / 0)

In [ ]:
# Close any existing plots
plt.close()

# Create a new figure and axis
fig, ax2 = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))

# Iterate over each run
for i, run in enumerate(runs):
    cutoffs = []
    timesteps = run.steps

    # Calculate the energy cutoff for each timestep
    for sel_ts in timesteps:
        bs = run.find_total_spectrum(sel_ts, species)

        cutoff = bs.calc_cutoff()
        cutoffs.append(cutoff)

    # Plot the energy cutoff vs. timestep
    ax2.plot(timesteps, cutoffs, label=f'Run {run.code}', color=colors[i])

# Set axis labels and title
ax2.set_xlabel('$t \Omega$')
ax2.set_ylabel('Energy Cutoff ($\gamma - 1$)')
ax2.set_title('Energy Cutoff vs. Timestep')
ax2.grid(visible=True)
ax2.legend()

# Show the plot
plt.show()